# section 2 — PyTorch Lightning, through MassSpecGym's own architecture

See `README.md` in this folder for the full plan and reasoning. Quick recap: this section is not a
generic Lightning tutorial — it's specifically about the `LightningModule` contract MassSpecGym's own
codebase is built on (`MassSpecGymModel` -> `RetrievalMassSpecGymModel` -> a concrete model), practiced
against their real data pipeline on their own small debug set (5 real spectra).

Before writing any code, read `massspecgym/models/base.py` and `massspecgym/models/retrieval/base.py`
in your installed environment (`.venv/Lib/site-packages/massspecgym/models/...`). This notebook assumes
you've done that — it won't re-explain what's in those files.


In [1]:
from pathlib import Path

import pytorch_lightning as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from pytorch_lightning.loggers import WandbLogger

from massspecgym.data import RetrievalDataset, MassSpecDataModule
from massspecgym.data.transforms import SpecBinner, MolFingerprinter
from massspecgym.models.retrieval import RetrievalMassSpecGymModel, RandomRetrieval, FingerprintFFNRetrieval

pl.seed_everything(0)

DEBUG_MGF = Path("../data/massspecgym_debug/example_5_spectra.mgf")
DEBUG_CANDIDATES = Path("../data/massspecgym_debug/example_5_spectra_candidates.json")
DEBUG_SPLIT = Path("../data/massspecgym_debug/example_5_spectra_split.tsv")


c:\Users\YarivBNB03\code\metabolomics-ml-curriculum\.venv\Lib\site-packages\lightning_fabric\__init__.py:41: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
Seed set to 0


## Phase 0 — orientation

Build the real MassSpecGym data pipeline for the retrieval task: `RetrievalDataset` (spectra + candidate
sets) wrapped by `MassSpecDataModule` (train/val/test splits + dataloaders), with `SpecBinner` as the
`spec_transform` (bins each spectrum into a fixed-length intensity vector — the same representation
`FingerprintFFNRetrieval` expects) and `MolFingerprinter` as the `mol_transform` (turns each molecule into
a binary fingerprint of `fp_size` bits — this becomes both the training target and the representation
candidates get scored in).

Look up the constructor signatures for `RetrievalDataset` and `MassSpecDataModule` (both importable
directly; check their docstrings/type hints) rather than guessing the argument names.


In [ ]:
FP_SIZE = 2048  # fingerprint size — also MyFingerprintRetrieval's output dimension later

# TODO: construct the dataset (pth=DEBUG_MGF, spec_transform=SpecBinner(), mol_transform=MolFingerprinter(fp_size=FP_SIZE),
# candidates_pth=DEBUG_CANDIDATES) and wrap it in a MassSpecDataModule (split_pth=DEBUG_SPLIT, batch_size=2)
dataset = ...
data_module = ...


Inspect one raw sample (`dataset[0]`) and one batched sample from the data module before building any
model. What keys does a sample dict have? What shape is `spec`? What shape is `mol`? `candidates` and
`batch_ptr` describe a variable number of candidate molecules per sample, concatenated rather than padded
— why might that representation be more natural here than padding to a fixed number of candidates?


In [ ]:
sample = dataset[0]
# TODO: print sample.keys(), and the shape of sample['spec'] / sample['mol']

data_module.prepare_data()
data_module.setup()
batch = next(iter(data_module.train_dataloader()))
# TODO: print batch.keys(), and the shape/dtype of batch['spec'], batch['mol'], batch['candidates'], batch['batch_ptr']


## Phase 1 — implement a retrieval model against their scaffolding

`RetrievalMassSpecGymModel` (which you read in `massspecgym/models/retrieval/base.py`) already implements
`on_batch_end` — it computes hit_rate@k, MCES@1, and calls `self.log(...)` for all of it. You do not need
to log anything yourself. What it needs from you is one method: `step(self, batch, stage)`, returning
`{"loss": ..., "scores": ...}`.

Design: a small feedforward network (`in_channels` = number of bins from `SpecBinner`, `out_channels` =
`FP_SIZE`) that maps a binned spectrum to a predicted fingerprint (values in `[0, 1]` — what activation
gives you that?). Compare the predicted fingerprint against `batch["mol"]` for the loss (cosine similarity
is a reasonable choice for comparing two fingerprint-like vectors — `torch.nn.functional.cosine_similarity`
does NOT give you a loss directly, since higher cosine similarity is better; think about how to turn it
into something to minimize).

The harder piece: turning your one predicted fingerprint per spectrum into `scores` against every
candidate. Each sample has a different number of candidates, concatenated together with `batch_ptr`
telling you how many candidates belong to each sample. Look up `torch.repeat_interleave` — what would
repeating each predicted fingerprint according to `batch_ptr` before comparing against `batch["candidates"]`
give you?

Do not look at `massspecgym/models/retrieval/fingerprint_ffn.py` until you've implemented this yourself —
compare afterward, not before.


In [ ]:
class MyFingerprintRetrieval(RetrievalMassSpecGymModel):
    def __init__(self, in_channels: int, out_channels: int, hidden_channels: int = 512, **kwargs):
        super().__init__(**kwargs)
        pass

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        pass

    def step(self, batch: dict, stage=None) -> dict:
        pass


## Phase 2 — train it with their `Trainer` + `WandbLogger`

`WandbLogger` is a different idiom from calling `wandb.init()`/`wandb.log()` yourself: you hand it to the
`Trainer`, and every `self.log(...)` call anywhere in the model (including the ones already happening
inside `RetrievalMassSpecGymModel.on_batch_end`, which you didn't have to write) gets forwarded to it
automatically. Requires having run `wandb login` at least once.

Real gotcha, taken directly from MassSpecGym's own demo notebook: calling `trainer.validate()` *before*
any `trainer.fit()` call requires manually calling `data_module.prepare_data()` and `data_module.setup()`
first — normally `.fit()`/`.test()` handle this for you, `.validate()` alone doesn't.


In [ ]:
model = MyFingerprintRetrieval(in_channels=sample["spec"].shape[-1], out_channels=FP_SIZE)

# TODO: construct a WandbLogger (give it a project name), then a pl.Trainer using it as `logger`
# (accelerator="cpu", max_epochs=..., since this is 5 spectra it should train almost instantly)
wandb_logger = ...
trainer = ...

# TODO: validate before fit (remember the prepare_data/setup gotcha above), then fit, then test


## Phase 3 (stretch, only if time) — compare against their own baselines

Run `RandomRetrieval` and `FingerprintFFNRetrieval` (both pre-built, imported above) against the same
`data_module` and compare their `hit_rate@k` / loss against your own model's, as a sanity check that
`MyFingerprintRetrieval` is behaving reasonably rather than just running without crashing.


In [ ]:
# Optional: instantiate RandomRetrieval() and FingerprintFFNRetrieval(in_channels=..., out_channels=FP_SIZE),
# run trainer.test(...) for each against the same data_module, and compare metrics against your own model.


## Reflection

- **What I built:**
- **What I learned about the MassSpecGym / Lightning contract:**
- **Where I was stuck, and for how long:**
